In [41]:
import pandas as pd
import numpy as np
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import nbformat 
import kaleido

In [16]:
# cargo CSV precio del bitcoin
btc = pd.read_csv("../data/btc.csv")
# elimino fila 0, donde aparece btc-usd
btc = btc[btc["Close"] != "BTC-USD"]

# convierto Date a datetime
btc['Date'] = pd.to_datetime(btc['Date'], errors='coerce')

# convierto las demás columnas a numéricas (float)
numeric_cols = ['Close', 'High', 'Low', 'Open', 'Volume']
for col in numeric_cols:
    btc[col] = pd.to_numeric(btc[col], errors='coerce')

# reviso cómo quedaron
btc.info()
btc.isnull().sum()

# primera descripcion
btc.describe()
btc.head()

<class 'pandas.DataFrame'>
RangeIndex: 3056 entries, 1 to 3056
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    3056 non-null   datetime64[us]
 1   Close   3056 non-null   float64       
 2   High    3056 non-null   float64       
 3   Low     3056 non-null   float64       
 4   Open    3056 non-null   float64       
 5   Volume  3056 non-null   int64         
dtypes: datetime64[us](1), float64(4), int64(1)
memory usage: 143.4 KB


,Date,Close,High,Low,Open,Volume
1,2018-02-01,9170.540039,10288.799805,8812.280273,10237.299805,9959400448
2,2018-02-02,8830.750000,9142.280273,7796.490234,9142.280273,12726899712
3,2018-02-03,9174.910156,9430.750000,8251.629883,8852.120117,7263790080
4,2018-02-04,8277.009766,9334.870117,8031.220215,9175.700195,7073549824
5,2018-02-05,6955.270020,8364.839844,6756.680176,8270.540039,9285289984


In [17]:
# Agrego columnas calculadas
btc['Daily_Change'] = btc['Close'] - btc['Open'] #Diferencia entre el precio de cierre y de apertura del día.
btc['Volatility'] = btc['High'] - btc['Low'] #Diferencia entre el valor máximo y mínimo del día.
btc['Pct_Change'] = btc['Close'].pct_change() #variación porcentual del precio de cierre respecto al día anterior.
btc['Volume_Change_pct'] = btc["Volume"].pct_change() #Variación porcentual del volumen de transacciones respecto al día anterior.
btc['SMA_7'] = btc["Close"].rolling(7).mean() #Promedio/media móvil a 7 días. Tendencia a corto plazo.
btc['SMA_30'] = btc["Close"].rolling(30).mean() #Promedio/media móvil a 30 días. Tendencia a largo plazo.
btc["Rolling_volatility_30"] = btc["Pct_Change"].rolling(30).std() #desviacion los ultimos 30 dias
lags = [1, 2, 3, 7]  # días anteriores

# Crear las columnas de lags
for lag in lags:
    btc[f'BTC_Close_t-{lag}'] = btc['Close'].shift(lag)


btc.head()
btc.tail(10)

,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,SMA_7,SMA_30,Rolling_volatility_30,BTC_Close_t-1,BTC_Close_t-2,BTC_Close_t-3,BTC_Close_t-7
3047,2026-06-05,60922.667969,63901.515625,59108.917969,63807.691406,71465606706,-2885.023438,4792.597656,-0.045123,0.120148,67728.080357,75548.714844,0.019127,63801.574219,64014.367188,66703.656250,73372.523438
3048,2026-06-06,60867.414062,61491.703125,59496.625000,60924.476562,30898464957,-57.062500,1995.078125,-0.000907,-0.567646,65887.020089,74910.628906,0.019128,60922.667969,63801.574219,64014.367188,73754.835938
3049,2026-06-07,63239.519531,64128.042969,60724.066406,60866.941406,36006496194,2372.578125,3403.976562,0.038972,0.165317,64409.853237,74345.720703,0.020954,60867.414062,60922.667969,63801.574219,73579.687500
3050,2026-06-08,63090.589844,64185.765625,62384.667969,63244.085938,34133252457,-153.496094,1801.097656,-0.002355,-0.052025,63234.255580,73759.928125,0.020822,63239.519531,60867.414062,60922.667969,71319.773438
3051,2026-06-09,61643.781250,63486.089844,60756.687500,63092.890625,40144449754,-1449.109375,2729.402344,-0.022932,0.176110,62511.416295,73076.756510,0.020387,63090.589844,63239.519531,60867.414062,66703.656250
3052,2026-06-10,61449.289062,62788.277344,60788.023438,61643.207031,27493506779,-193.917969,2000.253906,-0.003155,-0.315136,62144.976562,72400.789583,0.020404,61643.781250,63090.589844,63239.519531,64014.367188
3053,2026-06-11,63561.054688,63851.949219,61447.968750,61448.878906,29316677897,2112.175781,2403.980469,0.034366,0.066313,62110.616629,71836.908333,0.021859,61449.289062,61643.781250,63090.589844,63801.574219
3054,2026-06-12,63543.199219,64334.015625,62778.792969,63547.441406,26881868307,-4.242188,1555.222656,-0.000281,-0.083052,62484.978237,71312.444401,0.021854,63561.054688,61449.289062,61643.781250,60922.667969
3055,2026-06-13,64421.324219,64700.878906,63431.320312,63541.515625,16956245530,879.808594,1269.558594,0.013819,-0.369231,62992.679688,70758.113542,0.021508,63543.199219,63561.054688,61449.289062,60867.414062
3056,2026-06-14,64074.820312,64631.433594,63856.761719,64412.648438,17298849792,-337.828125,774.671875,-0.005379,0.020205,63112.008371,70258.418229,0.021266,64421.324219,63543.199219,63561.054688,63239.519531


In [18]:
# Cargar el índice Fear and Greed
fear_greed = pd.read_csv("../data/fear_greed.csv")

# fear_greed = fear_greed.drop(columns=['time_until_update'])
fear_greed = fear_greed.rename(columns={'date': 'Date'})

# Convertir a datetime desde formato día-mes-año
fear_greed['Date'] = pd.to_datetime(
    fear_greed['Date'].astype(str).str.strip(),
    format='%d-%m-%Y',
    errors='coerce'
)

fear_greed['Date'] = fear_greed['Date'].dt.tz_localize(None)
fear_greed['Date'] = fear_greed['Date'].dt.normalize()
fear_greed = fear_greed.sort_values(by='Date', ascending=True)
fear_greed = fear_greed.reset_index(drop=True)

# Mostrar información y primeros registros
fear_greed.describe()
fear_greed.info()
fear_greed.tail()


<class 'pandas.DataFrame'>
RangeIndex: 3052 entries, 0 to 3051
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                3052 non-null   datetime64[us]
 1   fng_value           3052 non-null   float64       
 2   fng_classification  3052 non-null   str           
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 71.7 KB


,Date,fng_value,fng_classification
3047,2026-06-10,9.0,Extreme Fear
3048,2026-06-11,12.0,Extreme Fear
3049,2026-06-12,12.0,Extreme Fear
3050,2026-06-13,13.0,Extreme Fear
3051,2026-06-14,18.0,Extreme Fear


In [19]:
# Agrego columnas calculadas:
fear_greed['fng_diff_day'] = fear_greed['fng_value'].diff()
fear_greed['fng_SMA_7'] = fear_greed['fng_value'].rolling(7).mean()
fear_greed['fng_SMA_30'] = fear_greed['fng_value'].rolling(30).mean()
fear_greed['fng_trend'] = fear_greed['fng_SMA_7'] - fear_greed['fng_SMA_30']
fear_greed.head(100)

,Date,fng_value,fng_classification,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend
0,2018-02-01,30.0,Fear,NaN,NaN,NaN,NaN
1,2018-02-02,15.0,Extreme Fear,-15.0,NaN,NaN,NaN
2,2018-02-03,40.0,Fear,25.0,NaN,NaN,NaN
3,2018-02-04,24.0,Extreme Fear,-16.0,NaN,NaN,NaN
4,2018-02-05,11.0,Extreme Fear,-13.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
95,2018-05-10,63.0,Greed,10.0,60.000000,42.033333,17.966667
96,2018-05-11,41.0,Fear,-22.0,57.857143,42.833333,15.023810
97,2018-05-12,44.0,Fear,3.0,55.142857,43.600000,11.542857
98,2018-05-13,40.0,Fear,-4.0,51.285714,44.333333,6.952381


In [20]:
full_data = pd.merge_asof(btc.sort_values('Date'),
                          fear_greed.sort_values('Date'),
                          on = 'Date',
                          direction='backward')
full_data.head(10)

,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,...,BTC_Close_t-1,BTC_Close_t-2,BTC_Close_t-3,BTC_Close_t-7,fng_value,fng_classification,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend
0,2018-02-01,9170.540039,10288.799805,8812.280273,10237.299805,9959400448,-1066.759766,1476.519531,NaN,NaN,...,NaN,NaN,NaN,NaN,30.0,Fear,NaN,NaN,NaN,NaN
1,2018-02-02,8830.750000,9142.280273,7796.490234,9142.280273,12726899712,-311.530273,1345.790039,-0.037052,0.277878,...,9170.540039,NaN,NaN,NaN,15.0,Extreme Fear,-15.0,NaN,NaN,NaN
2,2018-02-03,9174.910156,9430.750000,8251.629883,8852.120117,7263790080,322.790039,1179.120117,0.038973,-0.429257,...,8830.750000,9170.540039,NaN,NaN,40.0,Fear,25.0,NaN,NaN,NaN
3,2018-02-04,8277.009766,9334.870117,8031.220215,9175.700195,7073549824,-898.690430,1303.649902,-0.097865,-0.026190,...,9174.910156,8830.750000,9170.540039,NaN,24.0,Extreme Fear,-16.0,NaN,NaN,NaN
4,2018-02-05,6955.270020,8364.839844,6756.680176,8270.540039,9285289984,-1315.270020,1608.159668,-0.159688,0.312678,...,8277.009766,9174.910156,8830.750000,NaN,11.0,Extreme Fear,-13.0,NaN,NaN,NaN
5,2018-02-06,7754.000000,7850.700195,6048.259766,7051.750000,13999800320,702.250000,1802.440430,0.114838,0.507740,...,6955.270020,8277.009766,9174.910156,NaN,8.0,Extreme Fear,-3.0,NaN,NaN,NaN
6,2018-02-07,7621.299805,8509.110352,7236.790039,7755.490234,9169280000,-134.190430,1272.320312,-0.017114,-0.345042,...,7754.000000,6955.270020,8277.009766,NaN,36.0,Fear,28.0,23.428571,NaN,NaN
7,2018-02-08,8265.589844,8558.769531,7637.859863,7637.859863,9346750464,627.729980,920.909668,0.084538,0.019355,...,7621.299805,7754.000000,6955.270020,9170.540039,30.0,Fear,-6.0,23.428571,NaN,NaN
8,2018-02-09,8736.980469,8736.980469,7884.709961,8271.839844,6784820224,465.140625,852.270508,0.057030,-0.274098,...,8265.589844,7621.299805,7754.000000,8830.750000,44.0,Fear,14.0,27.571429,NaN,NaN
9,2018-02-10,8621.900391,9122.549805,8295.469727,8720.080078,7780960256,-98.179688,827.080078,-0.013172,0.146819,...,8736.980469,8265.589844,7621.299805,9174.910156,54.0,Neutral,10.0,29.571429,NaN,NaN


# Halving de Bitcoin

El halving es un evento programado que reduce a la mitad la recompensa por minar bloques, aproximadamente cada 4 años.

## Impacto en BTC

- **Menor oferta nueva:** ingresan menos bitcoins al mercado, lo que aumenta la escasez.
- **Efecto en el precio:** si la demanda se mantiene o crece, esa menor oferta puede impulsar subas en el precio.
- **Expectativa del mercado:** suele generar anticipación y mayor atención de inversores antes y después del evento.
- **Impacto en mineros:** reduce ingresos por bloque y puede afectar la rentabilidad de los mineros menos eficientes.

En ciclos anteriores, los halvings estuvieron seguidos por etapas de fuerte apreciación del precio, aunque no garantizan resultados futuros.

In [21]:
halvings = [pd.Timestamp("2012-11-28"),
            pd.Timestamp("2016-07-09"),
            pd.Timestamp("2020-05-11"),
            pd.Timestamp("2024-04-20")
           ]

rewards = [25, 12.5, 6.25, 3.125]

full_data["Is_Halving_Date"] = 0
full_data["Block_reward"] = np.nan

full_data.loc[full_data["Date"].isin(halvings), "Is_Halving_Date"] = 1

for i in range(len(halvings)):
    start = halvings[i]
    end = halvings[i+1] if i+1 < len(halvings) else full_data["Date"].max()

    mask = (full_data["Date"] >= start) & (full_data["Date"] < end)

    full_data.loc[mask, "Block_reward"] = rewards[i]


full_data.sample(10)

,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,...,BTC_Close_t-3,BTC_Close_t-7,fng_value,fng_classification,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend,Is_Halving_Date,Block_reward
673,2019-12-06,7546.996582,7546.996582,7392.175293,7450.561523,18104466307,96.435059,154.821289,0.013250,-0.037820,...,7320.145508,7761.243652,29.0,Fear,8.0,27.571429,31.933333,-4.361905,0,12.500
2689,2025-06-13,106090.968750,106182.546875,102822.023438,105924.593750,69550440846,166.375000,3360.523438,0.001529,0.268153,...,110257.234375,104390.343750,61.0,Greed,-10.0,64.428571,66.500000,-2.071429,0,3.125
1661,2022-08-20,21166.060547,21350.806641,20856.730469,20872.841797,27595671000,293.218750,494.076172,0.013819,-0.318787,...,23335.998047,24424.068359,29.0,Fear,-4.0,38.428571,35.033333,3.395238,0,6.250
2206,2024-02-16,52160.203125,52537.968750,51641.367188,51937.726562,28180567298,222.476562,896.601562,0.004268,-0.269259,...,49742.441406,47147.199219,72.0,Greed,0.0,73.142857,61.433333,11.709524,0,6.250
277,2018-11-05,6419.660156,6480.589844,6363.620117,6363.620117,4174800000,56.040039,116.969727,0.006827,-0.049025,...,6388.439941,6332.629883,42.0,Fear,1.0,35.285714,27.166667,8.119048,0,12.500
2799,2025-10-01,118648.929688,118648.929688,113981.398438,114057.593750,71328680132,4591.335938,4667.531250,0.040268,0.209241,...,112122.640625,113328.632812,49.0,Neutral,-1.0,41.571429,48.133333,-6.561905,0,3.125
2960,2026-03-11,70204.882812,71337.664062,68998.867188,69931.250000,45236859848,273.632812,2338.796875,0.003975,-0.162342,...,65969.781250,72710.578125,15.0,Extreme Fear,2.0,14.285714,10.700000,3.585714,0,3.125
100,2018-05-12,8504.889648,8664.860352,8223.500000,8441.440430,6821380096,63.449219,441.360352,0.007510,-0.196399,...,9325.179688,9858.150391,44.0,Fear,3.0,55.142857,43.600000,11.542857,0,12.500
2965,2026-03-16,74861.085938,74901.859375,72300.632812,72798.171875,55572438409,2062.914062,2601.226562,0.028454,0.985349,...,70968.265625,68402.382812,23.0,Extreme Fear,8.0,16.428571,12.166667,4.261905,0,3.125
2144,2023-12-16,42240.117188,42664.945312,41723.113281,41937.742188,14386729590,302.375000,941.832031,0.007402,-0.267457,...,42890.742188,43725.984375,67.0,Greed,-3.0,69.857143,70.100000,-0.242857,0,6.250


In [22]:
#Guardo el csv final para entrenal 
full_data.to_csv("../Data/full_data.csv",index=False)
full_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3056 entries, 0 to 3055
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Date                   3056 non-null   datetime64[us]
 1   Close                  3056 non-null   float64       
 2   High                   3056 non-null   float64       
 3   Low                    3056 non-null   float64       
 4   Open                   3056 non-null   float64       
 5   Volume                 3056 non-null   int64         
 6   Daily_Change           3056 non-null   float64       
 7   Volatility             3056 non-null   float64       
 8   Pct_Change             3055 non-null   float64       
 9   Volume_Change_pct      3055 non-null   float64       
 10  SMA_7                  3050 non-null   float64       
 11  SMA_30                 3027 non-null   float64       
 12  Rolling_volatility_30  3026 non-null   float64       
 13  BTC_Close_t-1 

In [27]:
full_data["Date"] = pd.to_datetime(full_data["Date"])  # asegurar tipo datetime
full_data= full_data.set_index("Date", drop=False)

In [46]:
# Seleciono las columnas del indice F&G para hacer el mapa de calor (correlacion)
fng_cols = ['fng_value', 'fng_diff_day', 'fng_SMA_7', 'fng_SMA_30', 'fng_trend']

# Columnas de precio BTC relevantes
btc_cols = ['Close', 'Daily_Change', 'Volatility', 'Pct_Change', 'Volume_Change_pct', 'SMA_7', 'SMA_30', 'Rolling_volatility_30', 'Block_reward']

# Selecciono solo las columnas de interés
corr_data = full_data[fng_cols + btc_cols].dropna()

# Calculo la correlación
corr_matrix = corr_data.corr()

# Heatmap interactivo con plotly
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Mapa de Calor - Correlación Fear & Greed vs BTC'
)
fig.show(renderer="browser")

In [49]:
# Crear figura
fig = go.Figure()

# Precio del Bitcoin
fig.add_trace(go.Scatter(
    x=full_data["Date"], 
    y=full_data["Close"],
    mode='lines', 
    name='BTC Price', 
    line=dict(color='blue')
))

# Block reward (como escalón)
fig.add_trace(go.Scatter(
    x=full_data["Date"], 
    y=full_data["Block_reward"],
    mode='lines', 
    name='Block Reward', 
    line=dict(color='red', dash='dash'),
    yaxis="y2"
))

# Puntos de halving
halving_points = full_data[full_data["Is_Halving_Date"] == 1]

fig.add_trace(go.Scatter(
    x=halving_points["Date"],
    y=halving_points["Close"],
    mode="markers+text",
    name="Halving",
    marker=dict(size=10, color="green", symbol="diamond"),
    text=halving_points["Close"].round(0),   # precio aproximado como etiqueta
    textposition="top center"
))

# Layout
fig.update_layout(
    title="Bitcoin Price vs Block Reward con Halvings",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    yaxis2=dict(title="Block Reward (BTC)", overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)

fig.show(renderer = "browser")

## Fear and Greed (índice de miedo y codicia)  
Es una medida del sentimiento del mercado, creada para el mercado de acciones, pero que también se aplica a criptomonedas. Su objetivo es reflejar si los inversores están dominados por el miedo o por la codicia, lo que puede influir en sus decisiones de compra o venta.  
El índice va de 0 a 100:  
- 0-25: Miedo extremo
- 26-49: Miedo
- 50-74: Codicia
- 75-100: Codicia extrema  

El gráfico refleja este patrón, puntos azules (Fear/Extreme Fear) coinciden con caídas del precio del bitcoin, y puntos rojos (Greed/Extreme Greed) coincide con picos de subida en el precio.

In [51]:
# Mapeo de colores para las categorías del Fear & Greed
colors = {
    'Extreme Fear': 'darkblue',
    'Fear': 'blue',
    'Neutral': 'orange',
    'Greed': 'red',
    'Extreme Greed': 'darkred'
}

df_plot = full_data[['Date', 'Close', 'fng_value', 'fng_classification']].dropna()

fig = go.Figure()

# Línea de precio de Bitcoin
fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Close'],
    mode='lines',
    name='Precio BTC',
    line=dict(color='black', width=2)
))

# Marcadores del Fear & Greed
fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['fng_value'],
    mode='markers',
    name='Fear & Greed',
    marker=dict(color=[colors[c] for c in df_plot['fng_classification']], size=10),
    text=df_plot['fng_classification'],  # Muestra la categoría al pasar el cursor
    hoverinfo='text+y'
))

# Layout
fig.update_layout(
    title='Precio de Bitcoin vs Fear & Greed',
    yaxis=dict(title='Precio BTC', side='left'),
    yaxis2=dict(title='Fear & Greed Value', overlaying='y', side='right', range=[0,100]),
    legend=dict(x=0.01, y=0.99)
)

fig.show(renderer = "browser")

**Visualización de tendencia en este último año**  
*SMA 7 encima de SMA 30 → tendencia alcista: el precio reciente está subiendo más rápido que la tendencia general.*   
*SMA 7 debajo de SMA 30 → tendencia bajista: el precio reciente está cayendo respecto a la tendencia general.*
    
Entre enero y marzo de 2026, el precio muestra una tendencia bajista marcada, cayendo desde un pico inicial superior a los 90 000 USD (tras el máximo del ciclo a fines del año pasado) hasta tocar un soporte crítico alrededor de los 60 000 USD.
Desde abril hasta mediados de mayo, se observa una fuerte recuperación temporal, con precios que logran repuntar con fuerza superando los 75 000 USD y buscando estabilizarse. 
En junio, el precio experimenta nuevos altibajos y una corrección que vuelve a testear la zona de soporte clave de los 60 000–65 000 USD, manteniendo al mercado a la expectativa.

*Cruce hacia arriba (SMA 7 cruza SMA 30 desde abajo): señal alcista y de compra potencial (“golden cross”).*  
*Cruce hacia abajo (SMA 7 cruza SMA 30 desde arriba): señal bajista y de venta potencial (“death cross”).*

A principios de abril, un cruce hacia arriba de la media corta valida el inicio del rebote técnico. Sin embargo, a finales de mayo, un nuevo cruce bajista anticipa la corrección y el aumento de volatilidad observados a principios de junio.  

**Tendencia actual (junio 2026):** El precio consolida cerca del soporte de los 66 000 USD, mostrando divergencias alcistas en temporalidades largas. La media de 7 días intenta estabilizarse sobre la de 30 días, lo que sugiere signos de un posible cambio de tendencia si logra romper las resistencias inmediatas.

In [52]:
fig = go.Figure()

df_short = full_data[full_data['Date'] >= '2026-01-01']

# Precio de cierre BTC
fig.add_trace(go.Scatter(
    x=df_short['Date'],
    y=df_short['Close'],
    mode='lines',
    name='BTC Close',
    line=dict(color='black', width=2)
))

# SMA 7 días
fig.add_trace(go.Scatter(
    x=df_short['Date'],
    y=df_short['SMA_7'],
    mode='lines',
    name='SMA 7 días',
    line=dict(color='blue', width=2, dash='dash')
))

# SMA 30 días
fig.add_trace(go.Scatter(
    x=df_short['Date'],
    y=df_short['SMA_30'],
    mode='lines',
    name='SMA 30 días',
    line=dict(color='red', width=2, dash='dash')
))

# Layout
fig.update_layout(
    title="Precio de Bitcoin con Promedios Móviles (7 y 30 días) desde ene-2025 a hoy",
    xaxis_title="Fecha",
    yaxis_title="Precio USD",
    template="plotly_white",
    width=900,
    height=500,
    hovermode="x unified"
)

fig.show(renderer ="browser")